# Part 3 — Exploratory analysis

In [1]:
import sys
sys.path.append("..")

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
FIGURES_DIR = Path("../figures")

In [2]:
## Loading Clean Data For Analysis
Express_df = pd.read_csv(PROCESSED_DIR / "ExpressMart_processed.csv", parse_dates=["order_datetime"])


In [3]:
Express_df.shape

(185652, 12)

In [4]:
Express_df.columns

Index(['order_id', 'product', 'quantity_ordered', 'price_each',
       'purchase_address', 'sales', 'order_datetime', 'month', 'hour', 'city',
       'state', 'city_state'],
      dtype='str')

In [5]:
Express_df.dtypes

order_id                     int64
product                        str
quantity_ordered             int64
price_each                 float64
purchase_address               str
sales                      float64
order_datetime      datetime64[us]
month                        int64
hour                         int64
city                           str
state                          str
city_state                     str
dtype: object

In [6]:
from src.summaries import summarize_group, summarize_numeric


Which City Has the highest Revenue

In [7]:
summarize_group(Express_df, "city_state")

,city_state,n_lines,units,revenue,mean_line_value
0,"San Francisco, CA",44650,50156,8252258.67,184.821023
1,"Los Angeles, CA",29563,33246,5447304.29,184.260876
2,"New York City, NY",24837,27887,4660526.52,187.644503
3,"Boston, MA",19897,22490,3657300.76,183.811668
4,"Atlanta, GA",14863,16584,2794199.07,187.996977
5,"Dallas, TX",14795,16705,2763659.01,186.796824
6,"Seattle, WA",14712,16533,2744896.03,186.575315
7,"Portland, OR",9996,11289,1869857.57,187.060581
8,"Austin, TX",9888,11135,1817544.35,183.813142
9,"Portland, ME",2451,2746,449321.38,183.321656


At what time of day do customers place the most orders?

In [8]:
by_hour = summarize_group(Express_df, "hour").sort_values("hour").reset_index(drop = True)
by_hour[["hour", "n_lines", "revenue"]]

,hour,n_lines,revenue
0,0,3890,710887.98
1,1,2338,458258.99
2,2,1238,232574.51
3,3,827,144683.04
4,4,848,162281.15
5,5,1318,230549.93
6,6,2481,447963.01
7,7,4002,744067.32
8,8,6252,1192309.20
9,9,8740,1638784.04


In [9]:
by_hour.sort_values("n_lines", ascending=False).reset_index(drop=True)[["hour", "n_lines", "revenue"]]

,hour,n_lines,revenue
0,19,12886,2411971.14
1,12,12573,2314359.85
2,11,12392,2296619.84
3,18,12263,2218374.01
4,20,12218,2280784.36
5,13,12115,2152369.98
6,14,10965,2082513.70
7,10,10929,1942988.08
8,21,10905,2040790.48
9,17,10884,2126553.97


In [10]:
summarize_numeric(Express_df, "sales")

mean       185.60
median      14.95
min          2.99
max       3400.00
Name: sales, dtype: float64

In [12]:
summarize_numeric(Express_df, "price_each")

mean       184.51
median      14.95
min          2.99
max       1700.00
Name: price_each, dtype: float64

In [13]:
Express_df["quantity_ordered"].value_counts().sort_index()

quantity_ordered
1    168261
2     13319
3      2919
4       805
5       236
6        80
7        24
8         5
9         3
Name: count, dtype: int64

In [15]:
(Express_df["quantity_ordered"].value_counts(normalize=True).sort_index()).round(1)

quantity_ordered
1    0.9
2    0.1
3    0.0
4    0.0
5    0.0
6    0.0
7    0.0
8    0.0
9    0.0
Name: proportion, dtype: float64

In [16]:
by_product = Express_df.groupby("product").agg(
    price=("price_each", "mean"),
    units=("quantity_ordered", "sum"),
)
by_product.sort_values("price")

,price,units
product,,
AAA Batteries (4-pack),2.99,30981
AA Batteries (4-pack),3.84,27615
USB-C Charging Cable,11.95,23927
Wired Headphones,11.99,20520
Lightning Charging Cable,14.95,23163
Bose SoundSport Headphones,99.99,13427
20in Monitor,109.99,4123
27in FHD Monitor,149.99,7538
Apple Airpods Headphones,150.00,15633


In [17]:
by_product[["price", "units"]].corr()

,price,units
price,1.000000,-0.601277
units,-0.601277,1.000000
